# Manufacturing OEE and downtime analysis
Synthetic data; run all cells in Google Colab. OEE = availability × performance × quality.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('https://raw.githubusercontent.com/YSayaovong/manufacturing-operations-analytics-portfolio/main/01-oee-downtime/data.csv')
assert (df.planned_minutes == df.runtime_minutes + df.downtime_minutes).all()
df['good_units'] = df.total_units - df.reject_units
df['availability'] = df.runtime_minutes / df.planned_minutes
df['performance'] = (df.ideal_cycle_minutes * df.total_units / df.runtime_minutes).clip(upper=1)
df['quality'] = df.good_units / df.total_units
df['oee'] = df.availability * df.performance * df.quality
print(df[['availability','performance','quality','oee']].mean().round(3))

In [ ]:
summary = df.groupby('machine').agg(planned=('planned_minutes','sum'), runtime=('runtime_minutes','sum'), downtime=('downtime_minutes','sum'), total=('total_units','sum'), good=('good_units','sum'))
summary['availability'] = summary.runtime/summary.planned
summary['quality'] = summary.good/summary.total
print(summary.round(3))
df.groupby('primary_downtime_reason').downtime_minutes.sum().sort_values().plot.barh(title='Downtime minutes by cause')
plt.tight_layout(); plt.show()

## Decision prompt
Identify the worst-performing machine and top downtime cause. Propose a maintenance or changeover experiment; compare OEE before and after using the same calculation. Aggregated OEE should be computed from underlying totals, not blindly averaging percentages.